# Gene ID hell, solved: a Claude tool that canonicalises gene IDs

**Companion notebook** for the rendered note at
👉 [tensoromics.com/notes/gene-id-resolver](https://tensoromics.com/notes/gene-id-resolver)

What you'll build:

1. A small Python function `resolve_gene_ids()` that hits [MyGene.info](https://mygene.info) and returns a clean table from a messy gene list.
2. The same function wired into the Anthropic Claude API as a **tool**, so you can ask gene-ID questions in plain English and Claude calls the function for you.
3. Three worked examples: enriching a DESeq2 results table, asking Claude conversationally, and translating a paper-supplement gene panel into your pipeline's ID namespace (with deprecated-alias flagging).

> **Editing workflow.** Code in this notebook is mirrored byte-for-byte in `tensoromics-web/content/notes/gene-id-resolver.mdx`. When you change one, port the change to the other.

## 0. Setup

In [ ]:
%pip install --quiet requests anthropic pandas

In [ ]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # or export in shell before launching Jupyter
# (only needed for Section 3 below; Sections 1 and 2 work without an API key)

## 1. The tool

`resolve_gene_ids()` accepts any mixed-format list (HGNC symbols, Ensembl IDs with or without version, Entrez integers as strings, RefSeq accessions, deprecated aliases) and returns one row per input with the canonical (symbol, Ensembl, Entrez, name) tuple, plus an alias warning when the input is a deprecated symbol.

We use [MyGene.info](https://mygene.info) — a free, no-key API maintained by Scripps that wraps NCBI, Ensembl, HGNC, MGI, and more in a single query interface.

In [ ]:
import requests
from typing import Literal

ORGANISM_TAX = {"human": 9606, "mouse": 10090}

def resolve_gene_ids(
    ids: list[str],
    organism: Literal["human", "mouse"] = "human",
) -> list[dict]:
    """Resolve a mixed list of gene identifiers to a canonical table.

    Input: any mix of HGNC symbols, Ensembl IDs (with or without version),
    Entrez integers as strings, RefSeq accessions, or deprecated aliases.

    Output: one dict per input, with canonical_symbol, ensembl_id,
    entrez_id, name, and any alias_warning.
    """
    # Strip Ensembl version suffixes (ENSG00000012048.20 -> ENSG00000012048).
    cleaned = [i.split(".")[0] if i.startswith("ENS") else i for i in ids]

    r = requests.post(
        "https://mygene.info/v3/query",
        data={
            "q": ",".join(cleaned),
            "scopes": "symbol,ensembl.gene,entrezgene,refseq,alias",
            "species": ORGANISM_TAX[organism],
            "fields": "symbol,ensembl.gene,entrezgene,name",
        },
        timeout=10,
    )
    r.raise_for_status()
    hits = r.json()

    out = []
    for original, hit in zip(ids, hits):
        if "notfound" in hit:
            out.append({
                "input": original,
                "canonical_symbol": None,
                "ensembl_id": None,
                "entrez_id": None,
                "name": None,
                "alias_warning": "not found",
            })
            continue

        # MyGene returns 'ensembl' as dict or list. Normalise.
        ens = hit.get("ensembl")
        ensembl_id = (
            ens["gene"] if isinstance(ens, dict)
            else (ens[0]["gene"] if isinstance(ens, list) and ens else None)
        )
        symbol = hit.get("symbol")

        out.append({
            "input": original,
            "canonical_symbol": symbol,
            "ensembl_id": ensembl_id,
            "entrez_id": str(hit.get("entrezgene")) if hit.get("entrezgene") else None,
            "name": hit.get("name"),
            "alias_warning": (
                f"'{original}' is a deprecated alias for '{symbol}'"
                if symbol and original.upper() != symbol.upper()
                   and not original.startswith(("ENS", "NM_", "NR_"))
                   and not original.isdigit()
                else None
            ),
        })
    return out

## 2. Worked example 1 — enrich a DESeq2 results table with gene symbols

You ran DESeq2 on counts from RSEM, so `gene_id` is versioned Ensembl. The table is precise but unreadable. Enrich it with `symbol`, `entrez_id`, and `name` so a wet-lab collaborator can read it and a downstream tool (IPA, Enrichr) can ingest it.

In [ ]:
import pandas as pd

# A small slice of a DESeq2 results table — gene_id is versioned Ensembl
# (what RSEM / STAR-Salmon emit by default).
deg = pd.DataFrame({
    "gene_id": [
        "ENSG00000012048.20",  # BRCA1
        "ENSG00000141510.15",  # TP53
        "ENSG00000139618.18",  # BRCA2
        "ENSG00000133703.13",  # KRAS
        "ENSG00000146648.21",  # EGFR
    ],
    "log2FoldChange": [2.31, -1.82, 1.47, 3.05, -2.41],
    "padj": [1.2e-12, 3.4e-9, 2.1e-7, 5.7e-15, 1.0e-10],
})

# One MyGene call for the whole batch, then merge into the DEG table.
resolved = pd.DataFrame(resolve_gene_ids(deg["gene_id"].tolist()))
deg = deg.merge(
    resolved[["input", "canonical_symbol", "entrez_id", "name"]],
    left_on="gene_id", right_on="input",
).drop(columns="input")
deg

**Expected**: the original `gene_id / log2FoldChange / padj` columns, joined with `canonical_symbol`, `entrez_id`, and `name`. One MyGene roundtrip for the whole batch (regardless of size). Now you can sort by `padj`, filter `abs(log2FoldChange) > 1`, and share. `symbol` makes it readable; `entrez_id` makes it portable to the next downstream tool.

## 3. Wire the tool to Claude (needs `ANTHROPIC_API_KEY`)

Two pieces: a small **declaration** that tells Claude the tool exists, and a short **loop** that runs the conversation.

In [ ]:
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing."

### 3a. Declare the tool

The `description` is the single most important field — Claude reads it to decide *whether* to call the tool. Specify when it applies, not just what it does.

In [ ]:
import json
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-4-7"  # sonnet / haiku also work and cost less

TOOLS = [{
    "name": "resolve_gene_ids",
    "description": (
        "Canonicalise gene identifiers (symbols, Ensembl, Entrez, RefSeq, "
        "aliases) into a clean (symbol, Ensembl, Entrez, name) table. Use "
        "whenever the user provides gene IDs and asks to resolve them."
    ),
    "input_schema": {
        "type": "object",
        "required": ["ids"],
        "properties": {
            "ids": {"type": "array", "items": {"type": "string"}, "description": "Gene identifiers to resolve."},
            "organism": {"type": "string", "enum": ["human", "mouse"], "default": "human"},
        },
    },
}]

### 3b. Run the conversation loop

Three steps, repeated until Claude is done: send the question, execute any tool call Claude requests, send the result back.

In [ ]:
def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    while True:
        resp = client.messages.create(
            model=MODEL, max_tokens=2048, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == "end_turn":
            return next(b.text for b in resp.content if b.type == "text")

        tu = next(b for b in resp.content if b.type == "tool_use")
        result = resolve_gene_ids(**tu.input)
        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": tu.id,
                "content": json.dumps(result),
            }]},
        ]

## 4. Worked example 2 — ask Claude conversationally

For prototyping or one-off lookups when you don't want to write pandas code.

In [ ]:
print(ask("What's the Ensembl ID for BRCA1, and what's its NCBI/Entrez ID?"))

**Expected** (wording varies): a prose answer that includes `ENSG00000012048` and `672`. Behind the scenes Claude called `resolve_gene_ids(ids=["BRCA1"])`, got the canonical record from MyGene, and wrote the answer.

The numeric IDs came from the API at request time, not from Claude's training memory. That is the distinction Skills can't deliver.

## 5. Worked example 3 — paper supplement → your pipeline's namespace

A collaborator shares a 6-gene HR/DDR panel from a paper's supplementary table — HGNC symbols only, mixed vintage. Your pipeline is indexed by Ensembl, so you need `ensembl_id` to filter your count matrix. The tool also flags deprecated aliases so you can audit the source.

In [ ]:
# Panel from a paper's supplement (HGNC symbols only, mixed-vintage).
# Two of these are deprecated aliases: p53 -> TP53, FANCD1 -> BRCA2.
panel = pd.DataFrame({
    "gene":     ["BRCA1", "p53", "ATM", "CHEK2", "FANCD1", "RAD51C"],
    "category": ["HR",    "TS",  "DDR", "DDR",   "HR",     "HR"],
})

resolved = pd.DataFrame(resolve_gene_ids(panel["gene"].tolist()))
panel = panel.merge(
    resolved[["input", "canonical_symbol", "ensembl_id", "alias_warning"]],
    left_on="gene", right_on="input",
).drop(columns="input")
panel

**Expected**: 6 rows. Four (`BRCA1`, `ATM`, `CHEK2`, `RAD51C`) resolve to themselves with `alias_warning=None`. Two — `p53` and `FANCD1` — resolve to `TP53` and `BRCA2` respectively, with the deprecated-alias warning populated.

`panel.ensembl_id` is now what you join against your count matrix. The two non-null `alias_warning` rows become your QC list — surface them in the report so the source pipeline (or paper's authors) can fix the naming.

## 6. Where to take this next

- **MCP server wrapper.** The same function can be exposed as an MCP server so it's available across every Claude Code session without re-importing in each project. Covered in the next note.
- **Cache results.** Wrap `resolve_gene_ids` in `functools.lru_cache` or `joblib.Memory` so repeated queries don't hit MyGene.
- **Auto species split.** If the input mixes human and mouse (rare-disease scenarios), branch on prefix conventions (mouse: `Brca1` title case; human: `BRCA1` upper) and call separately.
- **Compose with the Methods drafter Skill.** Once your gene table is canonical, hand the resolution provenance (MyGene, query date, species) to the Methods drafter Skill to write it up in your Methods section.